# ZImager integration: spectroscopy regression

The recorded `xshooter-spectroscopy.csv` predates the merge: ScopeSim `2780412e`, IRDB `8be3e2c9`. It contains full-frame checksums and integrated rates for a flat AB20 point source in both ZShooter detector modes and both X-shooter slit configurations. Atmospheric acquisition is omitted to hold the input fixed across revisions; the configured optical throughput, PSF, ADC and optical thermal emission remain active. These are optical rate images, before detector exposure and noise.

Run in a fresh kernel against the integrated stack. Each configuration is constructed afresh. A matching SHA-256 means every detector-plane sample is identical, including the spectrum and its spatial distribution. Runtime measurements include local computation only.

In [1]:
from time import perf_counter
import gc
import hashlib
import resource
import numpy as np
import pandas as pd
import astropy.units as u
from synphot import SourceSpectrum, ConstFlux1D
import scopesim as sim
from zs_scopesim_tools.helpers import configure_irdb_path, warning_prevent_sync_alt_ra_dec, disable_scopesim_progress_bars, disable_scopesim_top_level_catch

from zs_scopesim_tools.paths import repo_root

configure_irdb_path()
disable_scopesim_top_level_catch()
sim.utils.set_console_log_level("ERROR")
LABEL = "zimager_xshooter"
OUTPUT = repo_root() / "validation/zimager/data"
OUTPUT.mkdir(parents=True, exist_ok=True)
cases = [("ZS_SPEC", "ZShooter_v2", "SPEC"), ("ZS_ALT", "ZShooter_v2", "SPEC_ALTDETECT"), ("XS_faint", "XSHOOTER", "SPEC"), ("XS_high", "XSHOOTER", "SPEC")]
rows = []
for case, instrument, mode in cases:
    start = perf_counter()
    commands = sim.UserCommands(use_instrument=instrument, set_modes=[mode])
    for key, value in {"!OBS.airmass": 1.1, "!OBS.seeing": .65, "!OBS.brightness": "dark", "!OBS.mjdobs": 60310.25, "!SIM.random.seed": 913}.items():
        commands[key] = value
    if instrument == "ZShooter_v2":
        for key, value in {"!INST.vis_curr_slit": .7, "!INST.nir_curr_slit": .7, "!ATMO.ao_enabled": False, "!INST.include_diffuse_emissivity": False, "!SIM.spectral.spectral_bin_width": 1e-5}.items():
            commands[key] = value
    else:
        slits = ("1.0", "0.9", "0.9") if case == "XS_faint" else ("0.8", "0.9", "0.6")
        for arm, slit in zip(("uvb", "vis", "nir"), slits):
            commands[f"!INST.{arm}_curr_slit"] = slit
        for arm in ("uvb", "vis"):
            commands[f"!DET.biny_{arm}"] = 2 if case == "XS_faint" else 1
    warning_prevent_sync_alt_ra_dec(commands)
    # Fix the input spectrum across revisions and avoid network-dependent atmosphere acquisition.
    for document in commands.yaml_dicts:
        if "effects" in document:
            document["effects"] = [effect for effect in document["effects"] if effect["class"] not in ["TAPASTERCurve", "SkycalcTERCurve", "SkyBackgroundTERCurve", "PalaceAirglowEmission"]]
    train = sim.OpticalTrain(commands)
    setup_s = perf_counter() - start
    source = sim.Source(x=[0], y=[0], ref=[0], weight=[1], spectra=[SourceSpectrum(ConstFlux1D, amplitude=20*u.ABmag)])
    start = perf_counter()
    with disable_scopesim_progress_bars():
        train.observe(source, update=False)
    observe_s = perf_counter() - start
    for plane in train.image_planes:
        data = plane.data
        rows.append({"case": case, "plane": plane.id, "shape": str(data.shape), "sum_e_s": data.sum(dtype=np.float64), "peak_e_s": data.max(), "sha256": hashlib.sha256(data.tobytes()).hexdigest(), "setup_s": setup_s, "observe_s": observe_s})
    print(case, f"setup {setup_s:.2f}s, observe {observe_s:.2f}s, process peak {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 2**30:.2f} GiB", flush=True)
    del train, source, data, plane
    gc.collect()
pd.DataFrame(rows).to_csv(OUTPUT / f"{LABEL}-spectroscopy.csv", index=False)
display(pd.DataFrame(rows)[["case", "plane", "sum_e_s", "peak_e_s", "observe_s"]])

ZS_SPEC setup 4.85s, observe 47.07s, process peak 5.15 GiB
ZS_ALT setup 4.79s, observe 48.55s, process peak 6.13 GiB
XS_faint setup 1.14s, observe 84.46s, process peak 9.39 GiB
XS_high setup 1.32s, observe 71.81s, process peak 9.39 GiB


,case,plane,sum_e_s,peak_e_s,observe_s
0,ZS_SPEC,0,4734.705566,0.038387,47.069192
1,ZS_SPEC,1,6010.942676,0.042777,47.069192
2,ZS_SPEC,2,5115.297522,0.049914,47.069192
3,ZS_SPEC,3,5704.138773,0.061771,47.069192
4,ZS_SPEC,4,4761.252225,0.072037,47.069192
5,ZS_SPEC,5,724382.418424,3.308646,47.069192
6,ZS_ALT,0,2861.170516,0.024656,48.554773
7,ZS_ALT,1,5699.969404,0.039626,48.554773
8,ZS_ALT,2,4530.211131,0.042950,48.554773
9,ZS_ALT,3,5732.764899,0.088339,48.554773


In [3]:
baseline = pd.read_csv(OUTPUT / "xshooter-spectroscopy.csv", float_precision="round_trip").set_index(["case", "plane"])
current = pd.DataFrame(rows).set_index(["case", "plane"])
comparison = current[["sum_e_s", "peak_e_s"]].copy()
comparison["integrated_relative_change"] = current["sum_e_s"] / baseline["sum_e_s"] - 1
comparison["peak_relative_change"] = current["peak_e_s"] / baseline["peak_e_s"] - 1
comparison["all_pixels_identical"] = current["sha256"] == baseline["sha256"]
assert comparison["all_pixels_identical"].all()
comparison.to_csv(OUTPUT / "spectroscopy-comparison.csv")
display(comparison)
print("All 18 full detector-plane arrays match the pre-merge xshooter baseline exactly.")

sum_e_s  peak_e_s  integrated_relative_change  \
case     plane                                                        
ZS_SPEC  0        4734.705566  0.038387                         0.0   
         1        6010.942676  0.042777                         0.0   
         2        5115.297522  0.049914                         0.0   
         3        5704.138773  0.061771                         0.0   
         4        4761.252225  0.072037                         0.0   
         5      724382.418424  3.308646                         0.0   
ZS_ALT   0        2861.170516  0.024656                         0.0   
         1        5699.969404  0.039626                         0.0   
         2        4530.211131  0.042950                         0.0   
         3        5732.764899  0.088339                         0.0   
         4        5048.605307  0.102785                         0.0   
         5      785639.570517  4.763613                         0.0   
XS_faint 0        2944.227252  0.047769                         0.0   
         1        3029.590748  0.026271                         0.0   
         2      655764.653109  5.762910                         0.0   
XS_high  0        2622.933326  0.043097                         0.0   
         1        3029.590748  0.026271                         0.0   
         2      437879.072859  3.862418                         0.0   

                peak_relative_change  all_pixels_identical  
case     plane                                              
ZS_SPEC  0                       0.0                  True  
         1                       0.0                  True  
         2                       0.0                  True  
         3                       0.0                  True  
         4                       0.0                  True  
         5                       0.0                  True  
ZS_ALT   0                       0.0                  True  
         1                       0.0                  True  
         2                       0.0                  True  
         3                       0.0                  True  
         4                       0.0                  True  
         5                       0.0                  True  
XS_faint 0                       0.0                  True  
         1                       0.0                  True  
         2                       0.0                  True  
XS_high  0                       0.0                  True  
         1                       0.0                  True  
         2                       0.0                  True

All 18 full detector-plane arrays match the pre-merge xshooter baseline exactly.
